In [ ]:
import torch
from torch.utils import data
from torch import nn
from torchvision import datasets, transforms
import numpy as np
from torch.utils.data import Subset

In [ ]:
def get_k_fold_data(
    k: int,
    i: int,
    train_dataset,
    valid_dataset=None,
    seed: int = 42 #必须要一个种子 不然每次返回的随机序列不同 也就可能返回用同个数据当验证集
):
    if k < 2:
        raise ValueError("k必须大于等于2")

    if not 0 <= i < k:
        raise ValueError(f"i必须在0到{k - 1}之间")

    # 如果训练集和验证集使用相同的transform  传valid_dataset的目的是要用valid里的valid_transform
    if valid_dataset is None:
        valid_dataset = train_dataset

    if len(train_dataset) != len(valid_dataset):
        raise ValueError("两个Dataset必须对应同一批样本")

    num = len(train_dataset)

    # 生成0到num-1的随机样本编号
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(
        num,
        generator=generator
    ).tolist()  #生成一个序列 

    begin = i * num // k
    end = (i + 1) * num // k

    # 不再切分数据，而是切分编号
    valid_indices = indices[begin:end]
    train_indices = indices[:begin] + indices[end:] #列表可以直接相加

    # 根据编号构造子数据集
    train_data = Subset(
        train_dataset,
        train_indices
    )

    valid_data = Subset(   #因为train_dataset不是tensor 所以不能返回train_dataset[train_indices] 所以需要用这个函数来返回
        valid_dataset,
        valid_indices
    )

    return train_data, valid_data

In [ ]:
#这个k折交叉验证的写法也可以应用到之前的房价数据，这个函数需要传进来的trian_dataset=data.TensorDataset(*(features,labels)) 
#本质上 就是随机构造一个数列 然后切片这个数列 把切片后的子数列里的值取出来 去调用当作下标。这样就可以实现随机截取train_dataset的效果了。但是这个不是分层截取，本质上还是整体随机抽取，概率学的意义上是一样的